# 06 -- Comparison Baselines and Architecture Ablation

Three parts: (1) the R2-1 baselines (Isolation Forest/LOF, Graph Autoencoder,
IF+NSGA-II) and the destination-novelty feature restoration (Section 4.6 ablation);
(2) the Layer-1 architecture ablation (order/structure variants B/C, same-algorithm
variants D/E, imbalance-aware baselines EKS-4/5); (3) the informativeness table and
persistence-by-configuration figure. All rationale in `docs/methodology.md`.

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from awik.config import ANSWER_SET, MALICIOUS_USERS, BATCH_SIZE, DEFAULT_MAH_THRESH
from awik.features import ALL_FEATURES
from awik.evaluation import evaluate_against_answerset, enrichment
from awik.layer1 import run_awik_pipeline, run_kmeans_only, run_dbscan_only
from awik.layer1_ablation import (run_dbscan_then_kmeans, run_voting_ensemble,
                                    run_kmeans_then_kmeans, run_dbscan_then_dbscan)
from awik.baseline_unsupervised import run_unsupervised_baseline, run_isolation_forest, run_lof
from awik.neo4j_client import Neo4jConnection
from awik import io

df_master = io.load_df('01_df_master')
weeks_df = io.load_df('01_weeks_df')
N_MASTER = len(df_master)
df_threats = io.load_df('03_df_threats')
df_semua = io.load_df('03_df_semua')
eps_stage = io.load_json('02_eps_stage')
EPS_STAGE1, EPS_STAGE2 = eps_stage['EPS_STAGE1'], eps_stage['EPS_STAGE2']

POP = df_master['User_ID'].nunique()
h_l1 = evaluate_against_answerset(set(df_threats['User_ID'].unique()), MALICIOUS_USERS,
        'Layer 1: Magnitude (K-Means->DBSCAN->empirical Mahalanobis)')

## Section 4.6 ablation: restore destination-novelty feature (15 -> 17 dims)

Domain extraction is batched per-user (mirrors notebook 01's DFS pattern) -- a
per-week batching approach was tried first and was far slower (unindexed
timestamp filter forced a full scan per query).

In [ ]:
print("-> [Section 4.6 ablation] Restoring destination-novelty feature + delta (15->17 dim)...")
import os, time

db_conn = Neo4jConnection("bolt://localhost:7687", "neo4j", os.environ.get("NEO4J_PASSWORD", "changeme"))
CHECKPOINT_46 = '../data/interim/06_domain_novelty_checkpoint.csv'

if os.path.exists(CHECKPOINT_46):
    df_url = pd.read_csv(CHECKPOINT_46)
    print(f"   Loaded from checkpoint: {len(df_url):,} rows")
else:
    query_domains_batch = """
    UNWIND $batch AS uid
    MATCH (u:User {user_id: uid})-[r:VISITED_URL]->(url:URL)
    WITH u.user_id AS User_ID, r.timestamp.year AS Year, r.timestamp.week AS Week,
         toLower(url.url) AS raw_url
    WITH User_ID, Year, Week,
         split(replace(replace(raw_url, 'https://', ''), 'http://', ''), '/')[0] AS host
    RETURN DISTINCT User_ID, Year, Week, split(host, ':')[0] AS domain
    """
    all_users_46 = df_master['User_ID'].unique().tolist()
    n_batches_46 = (len(all_users_46) + BATCH_SIZE - 1) // BATCH_SIZE
    t0 = time.time()
    domain_batches = []
    for i in range(n_batches_46):
        batch = all_users_46[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
        df_b = db_conn.query_to_dataframe(query_domains_batch, {"batch": batch})
        if not df_b.empty:
            domain_batches.append(df_b)
        if (i + 1) % 10 == 0 or (i + 1) == n_batches_46:
            print(f"      batch {i+1}/{n_batches_46} | {time.time()-t0:.0f}s", flush=True)
    df_url = pd.concat(domain_batches, ignore_index=True) if domain_batches else \
        pd.DataFrame(columns=['User_ID','domain','Year','Week'])
    os.makedirs('../data/interim', exist_ok=True)
    df_url.to_csv(CHECKPOINT_46, index=False)
    print(f"   Checkpoint saved -> {CHECKPOINT_46}")

db_conn.close()
df_master_46 = df_master.copy()

if not df_url.empty:
    df_url = df_url.dropna(subset=['domain'])
    df_url['wk_idx'] = df_url['Year'].astype(int) * 100 + df_url['Week'].astype(int)
    first_seen = df_url.groupby(['User_ID', 'domain'])['wk_idx'].min().reset_index(name='first_wk_idx')
    novelty_counts = first_seen.groupby(['User_ID', 'first_wk_idx']).size().reset_index(name='n_new_url_domain')
    novelty_counts['Year'] = novelty_counts['first_wk_idx'] // 100
    novelty_counts['Week'] = novelty_counts['first_wk_idx'] % 100
    first_week_per_user = df_url.groupby('User_ID')['wk_idx'].min().rename('user_first_wk')
    novelty_counts = novelty_counts.merge(first_week_per_user, on='User_ID', how='left')
    novelty_counts.loc[novelty_counts['first_wk_idx'] == novelty_counts['user_first_wk'], 'n_new_url_domain'] = 0
    df_novelty = novelty_counts[['User_ID', 'Year', 'Week', 'n_new_url_domain']]
    df_master_46 = df_master_46.merge(df_novelty, on=['User_ID', 'Year', 'Week'], how='left')
else:
    print("   [WARNING] df_url is empty -- check schema (r.url vs url.url).")
    df_master_46['n_new_url_domain'] = 0
df_master_46['n_new_url_domain'] = df_master_46['n_new_url_domain'].fillna(0)

def compute_delta_46(s):
    return (s - s.shift(1).rolling(4, min_periods=1).mean()).fillna(0)

df_master_46 = df_master_46.sort_values(['User_ID', 'Year', 'Week']).reset_index(drop=True)
df_master_46['delta_new_domain'] = df_master_46.groupby('User_ID')['n_new_url_domain'].transform(compute_delta_46)

FITUR_46 = ALL_FEATURES + ['n_new_url_domain', 'delta_new_domain']
print(f"   Feature vector: {len(ALL_FEATURES)} -> {len(FITUR_46)} dimensions")

df_semua_46, df_threats_46 = run_awik_pipeline(df_master_46, FITUR_46, weeks_df)
l1_flagged_46 = set(df_threats_46['User_ID'])
recall_l1_46 = len(l1_flagged_46 & set(ANSWER_SET)) / len(ANSWER_SET)
print(f"   Layer 1 recall (17-dim): {recall_l1_46:.4f}  [15-dim baseline: see h_l1 above]")
print(f"   Layer 1 flagged population: {len(l1_flagged_46):,}")

In [ ]:
# Adaptive baseline: causal, with a 4th channel (delta_new_domain). Same
# expanding+shift(1) pattern as notebook 04. MIN_OBS=8 still gates email only,
# consistent with notebook 04; the new domain-novelty channel is not explicitly
# gated -- worth revisiting if it turns out to be sensitive to small samples the
# way ah/usb were (R2-3 finding).
MIN_OBS = 8
df_sorted_46 = df_master_46.sort_values(['User_ID', 'Year', 'Week']).reset_index(drop=True)
g46 = df_sorted_46.groupby('User_ID', sort=False)

for col, pfx in [('delta_afterhour', 'dah'), ('delta_usb', 'dusb'),
                  ('delta_email_size', 'dmail'), ('delta_new_domain', 'ddom')]:
    df_sorted_46[f'{pfx}_mean_c'] = g46[col].expanding().mean().shift(1).reset_index(level=0, drop=True)
    df_sorted_46[f'{pfx}_std_c']  = g46[col].expanding().std().shift(1).reset_index(level=0, drop=True)
df_sorted_46['n_obs_c'] = g46.cumcount()
df_sorted_46['email_scale_c'] = g46['email_mean_size'].expanding().mean().shift(1).reset_index(level=0, drop=True)
for pfx in ['dah', 'dusb', 'dmail', 'ddom']:
    df_sorted_46[f'{pfx}_std_c'] = df_sorted_46[f'{pfx}_std_c'].fillna(0)

lon_ah_46  = df_sorted_46['delta_afterhour']  > (df_sorted_46['dah_mean_c'].fillna(np.inf)  + 2*df_sorted_46['dah_std_c'].clip(lower=0.1))
lon_usb_46 = df_sorted_46['delta_usb']        > (df_sorted_46['dusb_mean_c'].fillna(np.inf) + 2*df_sorted_46['dusb_std_c'].clip(lower=0.1))
lon_dom_46 = df_sorted_46['delta_new_domain'] > (df_sorted_46['ddom_mean_c'].fillna(np.inf) + 2*df_sorted_46['ddom_std_c'].clip(lower=0.1))
enough_46 = df_sorted_46['n_obs_c'] >= MIN_OBS
floor_em_46 = (0.5*df_sorted_46['email_scale_c'].fillna(0)).clip(lower=1.0)
lon_email_46 = enough_46 & (df_sorted_46['delta_email_size'] >
                             (df_sorted_46['dmail_mean_c'].fillna(np.inf) + 2*df_sorted_46['dmail_std_c'].clip(lower=floor_em_46)))

any_surge_46 = (lon_ah_46 | lon_usb_46 | lon_email_46 | lon_dom_46)
df_surges_46 = df_sorted_46.loc[any_surge_46, ['User_ID', 'Year', 'Week']].copy()
df_surges_46['Year_Week'] = (df_surges_46['Year'].astype(int).astype(str) + '-W' +
                                df_surges_46['Week'].astype(int).astype(str).str.zfill(2))

state04 = io.load_json('04_layer2_state')
K_PERSIST = state04['k_persist']
if not df_surges_46.empty:
    per_user_46 = df_surges_46.groupby('User_ID').agg(surge_weeks=('Year_Week', 'nunique'))
    cand_persist_46 = set(per_user_46.index[per_user_46['surge_weeks'] >= K_PERSIST])
else:
    cand_persist_46 = set()
# Persistence signal only, for a quick estimate -- for a precise number replicate
# the full co-occurrence/burst logic from notebook 04 (awik.adaptive_baseline.target_users).
users_targeted_46 = cand_persist_46

print(f"Routing pool (T, 17-dim): {len(users_targeted_46)}")
print()
print("Compare recall_l1_46 and len(users_targeted_46) above against the manuscript's")
print("Section 4.6 claim, using the CAUSAL (notebook 04) baseline as the comparison point --")
print("not the old non-causal numbers from earlier drafts.")

## R2-1a: Isolation Forest & LOF (unsupervised anomaly baseline)

Standard unsupervised anomaly detectors, same 15-feature matrix, same per-week /
union-to-account protocol as Layer 1 -- for a fair comparison, not numbers from a
different setup. Not a reproduction of one specific paper; these are the baselines
that already appear in most of this literature (including Artioli et al. [7]).
`contamination=0.05` is the standard default, not tuned to the Answer Set.

In [ ]:
print("-> [R2-1a] Baseline: Isolation Forest & LOF...")

_, if_flagged = run_unsupervised_baseline(df_master, ALL_FEATURES, weeks_df, method="iforest")
if_tp = len(if_flagged & set(ANSWER_SET)); if_pop = len(if_flagged)
print(f"Isolation Forest : population={if_pop:,} | TP={if_tp}/6 | recall={if_tp/6:.4f}" +
      (f" | precision={if_tp/if_pop:.4%}" if if_pop else " | (empty)"))

_, lof_flagged = run_unsupervised_baseline(df_master, ALL_FEATURES, weeks_df, method="lof")
lof_tp = len(lof_flagged & set(ANSWER_SET)); lof_pop = len(lof_flagged)
print(f"LOF              : population={lof_pop:,} | TP={lof_tp}/6 | recall={lof_tp/6:.4f}" +
      (f" | precision={lof_tp/lof_pop:.4%}" if lof_pop else " | (empty)"))

print(f"\nEnrichment Isolation Forest: {enrichment(if_tp, if_pop, 6, POP):.2f}x  [AWIK Layer 2: 4.4x]")
print(f"Enrichment LOF             : {enrichment(lof_tp, lof_pop, 6, POP):.2f}x  [AWIK Layer 2: 4.4x]")
print()
print("NOTE: contamination=0.05 is the sklearn default, NOT tuned to the Answer Set")
print("(label-free, consistent with AWIK's own protocol).")

## R2-1b: Graph Autoencoder (graph method baseline)

Not a reproduction of Yumlembam et al. -- their pipeline proved infeasible to
reproduce in reasonable time (environment-specific code; feature extraction for the
29 malicious accounts alone took >5 hours before adding any benign sample). This is
instead a standard GAE (Kipf & Welling 2016, `torch_geometric.nn.GAE`) on the
shared-PC co-occurrence graph AWIK already uses for G1, with node features = mean of
the 15 AWIK behavioral features per account -- representing the "graph method"
category, evaluated under the same protocol.

In [ ]:
import os
print("-> [R2-1b] Baseline: Graph Autoencoder (graph method)...")
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GAE
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler

db_conn = Neo4jConnection("bolt://localhost:7687", "neo4j", os.environ.get("NEO4J_PASSWORD", "changeme"))
query_shared_pc = """
MATCH (u1:User)-[:LOGGED_ON_TO]->(pc:PC)<-[:LOGGED_ON_TO]-(u2:User)
WHERE u1.user_id < u2.user_id
RETURN u1.user_id AS User1, u2.user_id AS User2, count(*) AS weight
"""
df_edges = db_conn.query_to_dataframe(query_shared_pc)
db_conn.close()
print(f"   Shared-PC edges: {len(df_edges):,} account pairs")

df_node_feat = df_master.groupby("User_ID")[ALL_FEATURES].mean().reset_index()
all_users_g = sorted(df_node_feat["User_ID"].unique().tolist())
user_to_idx = {u: i for i, u in enumerate(all_users_g)}
X_node = StandardScaler().fit_transform(
    df_node_feat.set_index("User_ID").loc[all_users_g][ALL_FEATURES].values)
X_node = torch.tensor(X_node, dtype=torch.float)

edges = []
for _, row in df_edges.iterrows():
    u1, u2 = row["User1"], row["User2"]
    if u1 in user_to_idx and u2 in user_to_idx:
        i, j = user_to_idx[u1], user_to_idx[u2]
        edges.append([i, j]); edges.append([j, i])
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
print(f"   Final graph: {len(all_users_g):,} nodes, {edge_index.shape[1]:,} directed edges")

data = Data(x=X_node, edge_index=edge_index)

class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

torch.manual_seed(42)
model = GAE(GCNEncoder(X_node.shape[1], 32, 16))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print("   Training GAE (unsupervised, no labels used)...")
model.train()
for epoch in range(200):
    optimizer.zero_grad()
    z = model.encode(data.x, data.edge_index)
    loss = model.recon_loss(z, data.edge_index)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f"      epoch {epoch+1}/200 | loss={loss.item():.4f}")

model.eval()
with torch.no_grad():
    z = model.encode(data.x, data.edge_index)
    adj_pred = torch.sigmoid(torch.matmul(z, z.t()))
    adj_true = torch.zeros((len(all_users_g), len(all_users_g)))
    adj_true[data.edge_index[0], data.edge_index[1]] = 1.0
    recon_error = ((adj_pred - adj_true) ** 2).mean(dim=1).numpy()

df_gae_score = pd.DataFrame({"User_ID": all_users_g, "recon_error": recon_error})
thresh = df_gae_score["recon_error"].mean() + 2 * df_gae_score["recon_error"].std()
gae_flagged = set(df_gae_score.loc[df_gae_score["recon_error"] > thresh, "User_ID"])
gae_tp, gae_pop = len(gae_flagged & set(ANSWER_SET)), len(gae_flagged)
print(f"\nGraph Autoencoder : population={gae_pop:,} | TP={gae_tp}/6 | recall={gae_tp/6:.4f}" +
      (f" | precision={gae_tp/gae_pop:.4%}" if gae_pop else " | (empty)"))
print(f"Enrichment: {enrichment(gae_tp, gae_pop, 6, POP):.2f}x  [AWIK Layer 2: 4.4x]")
print()
print("NOTE: threshold mean+2std is label-free (same spirit as AWIK). Architecture")
print("(32-16 hidden dims, 200 epochs, lr=0.01) is a standard/default choice, not tuned.")

## R2-1c: Isolation Forest + NSGA-II (optimized unsupervised baseline)

Replaces vanilla Isolation Forest for the "optimized Isolation Forest methods"
category the reviewer named explicitly (citing Gumaste et al.). Not a reproduction
of that paper -- its full methodology wasn't accessible at time of writing (too
recent, not yet indexed) -- this is the standard "Isolation Forest tuned by NSGA-II"
pattern via `pymoo`. **Important**: NSGA-II here optimizes recall against the
Answer Set as one objective, so this baseline is supervised-tuned, fundamentally
different from AWIK's fully label-free design. State this explicitly wherever these
numbers are cited.

In [ ]:
print("-> [R2-1c] Baseline: Isolation Forest + NSGA-II (optimized)...")
try:
    from pymoo.core.problem import Problem
    from pymoo.algorithms.moo.nsga2 import NSGA2
    from pymoo.optimize import minimize
    from pymoo.operators.sampling.rnd import FloatRandomSampling
except ImportError:
    import sys as _sys, subprocess
    subprocess.run([_sys.executable, '-m', 'pip', 'install', 'pymoo'], check=True)
    from pymoo.core.problem import Problem
    from pymoo.algorithms.moo.nsga2 import NSGA2
    from pymoo.optimize import minimize
    from pymoo.operators.sampling.rnd import FloatRandomSampling

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

df_node_feat_if = df_master.groupby("User_ID")[ALL_FEATURES].mean().reset_index()
X_if = StandardScaler().fit_transform(df_node_feat_if[ALL_FEATURES].values)
users_if = df_node_feat_if["User_ID"].tolist()
answer_idx = set(i for i, u in enumerate(users_if) if u in ANSWER_SET)

class IFTuningProblem(Problem):
    # 4 IF hyperparameters, 2 objectives (both minimized): 1-recall, and flagged
    # population fraction (a false-positive-load proxy).
    def __init__(self):
        super().__init__(n_var=4, n_obj=2, n_constr=0,
                          xl=np.array([50,  0.2, 0.01, 0.5]),
                          xu=np.array([300, 1.0, 0.15, 1.0]))

    def _evaluate(self, X, out, *args, **kwargs):
        f1_list, f2_list = [], []
        for row in X:
            n_est, max_samp, contam, max_feat = int(round(row[0])), float(row[1]), float(row[2]), float(row[3])
            model = IsolationForest(n_estimators=n_est, max_samples=max_samp,
                                     contamination=contam, max_features=max_feat, random_state=42)
            pred = model.fit_predict(X_if)
            flagged = set(i for i, p in enumerate(pred) if p == -1)
            tp = len(flagged & answer_idx)
            recall = tp / len(answer_idx) if answer_idx else 0
            f1_list.append(1 - recall)
            f2_list.append(len(flagged) / len(users_if))
        out["F"] = np.column_stack([f1_list, f2_list])

print("   Running NSGA-II (40 population x 30 generations -- takes a few minutes)...")
algorithm = NSGA2(pop_size=40, sampling=FloatRandomSampling())
res = minimize(IFTuningProblem(), algorithm, ('n_gen', 30), seed=42, verbose=False)

F = res.F
best_idx = np.lexsort((F[:, 1], F[:, 0]))[0]  # sort by 1-recall first, then pop_frac
n_est, max_samp, contam, max_feat = [float(v) for v in res.X[best_idx]]
n_est = int(round(n_est))
print(f"   Chosen Pareto solution: n_estimators={n_est}, max_samples={max_samp:.3f}, "
      f"contamination={contam:.4f}, max_features={max_feat:.3f}")

model_final = IsolationForest(n_estimators=n_est, max_samples=max_samp,
                                contamination=contam, max_features=max_feat, random_state=42)
pred_final = model_final.fit_predict(X_if)
nsga_flagged = set(users_if[i] for i, p in enumerate(pred_final) if p == -1)
nsga_tp, nsga_pop = len(nsga_flagged & set(ANSWER_SET)), len(nsga_flagged)
print(f"\nIF+NSGA-II (optimized) : population={nsga_pop:,} | TP={nsga_tp}/6 | recall={nsga_tp/6:.4f}" +
      (f" | precision={nsga_tp/nsga_pop:.4%}" if nsga_pop else " | (empty)"))
print(f"Enrichment: {enrichment(nsga_tp, nsga_pop, 6, POP):.2f}x  [AWIK Layer 2: 4.4x]")
print()
print("NOTE: this baseline is SUPERVISED-TUNED (recall vs. Answer Set is an NSGA-II")
print("objective) -- state this explicitly, don't equate it with the other unsupervised baselines.")

## Ablation study: spatial layer only (K-Means alone, DBSCAN alone)

In [ ]:
print("-> [EKS-2] K-Means alone...")
_, df_eks2 = run_kmeans_only(df_master, ALL_FEATURES, weeks_df)
h_eks2 = evaluate_against_answerset(set(df_eks2['User_ID'].unique()), MALICIOUS_USERS, 'EKS-2: K-Means alone')

print("-> [EKS-3] Global DBSCAN alone...")
_, df_eks3 = run_dbscan_only(df_master, ALL_FEATURES, weeks_df)
h_eks3 = evaluate_against_answerset(set(df_eks3['User_ID'].unique()), MALICIOUS_USERS, 'EKS-3: Global DBSCAN alone')

print()
print("n=6 is too small to claim recall superiority. Main argument: signal")
print("persistence (chronic vs. scattered), not recall.")
df_ablation = pd.DataFrame([h_l1, h_eks2, h_eks3])[['Label','Recall','Precision','F1','TP','FN','FP']]
for h in [h_l1, h_eks2, h_eks3]:
    print(f"  {h['Label']}\n    TP: {h['TP_list']} | FN: {h['FN_list']}")
df_ablation

In [ ]:
def persistensi(df_t, label):
    c = df_t.groupby('User_ID')['Year_Week'].nunique()
    n = len(c)
    return {'Method': label, 'Total_users': n,
            'Mean_weeks': round(c.mean(), 2) if n else 0,
            'Median_weeks': c.median() if n else 0,
            'Chronic_>=5x': (f"{(c>=5).sum()} ({(c>=5).mean()*100:.1f}%)" if n else '-')}, c

r1, c1 = persistensi(df_threats, 'EKS-1 AWIK')
r2, c2 = persistensi(df_eks2, 'EKS-2 K-Means')
r3, c3 = persistensi(df_eks3, 'EKS-3 DBSCAN')

print("SIGNAL PERSISTENCE (Chronic vs. Scattered):")
persistence_df = pd.DataFrame([r1, r2, r3])
print()
print("Weeks flagged, per CERT actor:")
for actor in sorted(MALICIOUS_USERS):
    vals = {l: c.get(actor, 0) for l, c in [('EKS-1', c1), ('EKS-2', c2), ('EKS-3', c3)]}
    print(f"  {actor:10s}: " + " | ".join(f"{k}={v}x" for k, v in vals.items()))
persistence_df

## Order & structure ablation: two-stage sequential (A) vs. reversed (B) vs. voting (C)

In [ ]:
_, df_B = run_dbscan_then_kmeans(df_master, ALL_FEATURES, weeks_df)
_, df_C = run_voting_ensemble(df_master, ALL_FEATURES, weeks_df)

def profil(df_flag, label):
    a = df_flag[df_flag['User_ID'].isin(MALICIOUS_USERS)]
    dw = a.groupby('User_ID')['Year_Week'].nunique()
    caught = sorted(set(a['User_ID'].unique()) & set(MALICIOUS_USERS))
    tot = len(df_flag); fr = tot/len(df_semua)*100
    fl = set(df_flag['User_ID'].unique()); tp = len(set(MALICIOUS_USERS)&fl)
    enr = enrichment(tp, len(fl), 6, POP)
    print(f"{label:22s}: actors {len(caught)}/6 {caught} | det-weeks median "
          f"{dw.median() if len(dw) else 0:.0f} (n={len(dw)}) | flag {tot:,} ({fr:.2f}%) | enrich {enr:.1f}x")

print("=== ABLATION: two-stage sequential (A) vs DB->KM (B) vs voting (C) ===")
profil(df_threats, 'A: two-stage KM->DB')
profil(df_B, 'B: DB->KM')
profil(df_C, 'C: voting >=2/3')

def per_aktor_dw(df_flag, label):
    dw = df_flag[df_flag['User_ID'].isin(MALICIOUS_USERS)].groupby('User_ID')['Year_Week'].nunique().to_dict()
    print(f"{label:20s}: " + " | ".join(f"{u}={dw.get(u,0)}" for u in sorted(MALICIOUS_USERS)))

print("\nDETECTION-WEEKS PER ACTOR (0 = never detected):")
per_aktor_dw(df_threats, 'A two-stage')
per_aktor_dw(df_B, 'B DB->KM')
per_aktor_dw(df_C, 'C voting')
print("\nVoting (C) is inferior (det-weeks ~1, 2/6). Two-stage sequential (A/B) is far")
print("more persistent -- hence 'two-stage spatial clustering', not 'ensemble'.")

## Same-algorithm-twice ablation: double K-Means (D) and double DBSCAN (E)

In [ ]:
_, df_D = run_kmeans_then_kmeans(df_master, ALL_FEATURES, weeks_df)
_, df_E = run_dbscan_then_dbscan(df_master, ALL_FEATURES, weeks_df)

print("=== Five variants, summary ===")
profil(df_threats, 'A: KM->DB (AWIK design)')
profil(df_B, 'B: DB->KM')
profil(df_C, 'C: voting >=2/3')
profil(df_D, 'D: KM->KM (double K-Means)')
profil(df_E, 'E: DB->DB (double DBSCAN)')

print("\nDETECTION-WEEKS PER ACTOR, all variants:")
per_aktor_dw(df_threats, 'A KM->DB')
per_aktor_dw(df_B, 'B DB->KM')
per_aktor_dw(df_C, 'C voting')
per_aktor_dw(df_D, 'D KM->KM')
per_aktor_dw(df_E, 'E DB->DB')

## DBSCAN->DBSCAN with label-free parameters -- REJECTED

Reuses `run_dbscan_then_dbscan` with `EPS_STAGE1`/`EPS_STAGE2` from notebook 02
(instead of the arbitrary `eps2_ratio=0.5` used for `df_E` above). `df_E` above is
left untouched -- it stands as the record that the first-pass parameter was a rough
guess. Result: flagging rate explodes to 21.4%; enrichment drops despite higher TP --
a detector "winning" recall by flagging almost everyone. Design A stays final.

In [ ]:
EPS2_RATIO_DERIVED = EPS_STAGE2 / EPS_STAGE1
print(f"eps stage-1 (label-free): {EPS_STAGE1:.4f} | eps stage-2 (label-free): {EPS_STAGE2:.4f} "
      f"(ratio to stage-1: {EPS2_RATIO_DERIVED:.4f})")

_, df_E_v2 = run_dbscan_then_dbscan(df_master, ALL_FEATURES, weeks_df, eps=EPS_STAGE1, eps2_ratio=EPS2_RATIO_DERIVED)

h_E_v2 = evaluate_against_answerset(set(df_E_v2['User_ID'].unique()), MALICIOUS_USERS,
        'E-v2: DBSCAN->DBSCAN (label-free, REJECTED)')
h_E_old = evaluate_against_answerset(set(df_E['User_ID'].unique()), MALICIOUS_USERS,
        'E-old: DBSCAN->DBSCAN (eps2_ratio=0.5, arbitrary)')
h_A = evaluate_against_answerset(set(df_threats['User_ID'].unique()), MALICIOUS_USERS,
        'A: K-Means->DBSCAN (final design)')

print()
print(pd.DataFrame([h_A, h_E_old, h_E_v2])[['Label', 'Recall', 'Precision', 'F1', 'TP', 'FN', 'FP']])
print()
profil(df_threats, 'A: K-Means->DBSCAN (final)')
profil(df_E, 'E-old: DBSCAN->DBSCAN (arbitrary eps2)')
profil(df_E_v2, 'E-v2: DBSCAN->DBSCAN (label-free eps2, REJECTED)')
print()
per_aktor_dw(df_threats, 'A (final)')
per_aktor_dw(df_E, 'E-old (arbitrary)')
per_aktor_dw(df_E_v2, 'E-v2 (label-free)')

df_E_v2.to_csv('../results/tables/AWIK_E_v2_DBDB_Rejected.csv', index=False)
print("\n=> CONCLUSION: DBSCAN->DBSCAN rejected as a Layer 1 replacement. Design A")
print("   (K-Means->DBSCAN) is the only one that stays reasonable once EVERY")
print("   parameter is derived label-free consistently.")

## Imbalance-aware baselines: Isolation Forest & LOF (`contamination='auto'`)

`contamination='auto'` (sklearn default) -- NOT set to the true 6/4000, that would be
far worse label leakage than anything rejected so far. Result: algorithms designed for
rare anomalies do NOT automatically win when run naively per-week without a
persistence-aggregation layer like AWIK's Cell 6/notebook 04 -- concrete evidence that
AWIK's contribution is in its routing architecture, not just the Layer-1 detector.

In [ ]:
_, df_iso = run_isolation_forest(df_master, ALL_FEATURES, weeks_df)
_, df_lof = run_lof(df_master, ALL_FEATURES, weeks_df)

h_iso = evaluate_against_answerset(set(df_iso['User_ID'].unique()), MALICIOUS_USERS, 'EKS-4: Isolation Forest')
h_lof = evaluate_against_answerset(set(df_lof['User_ID'].unique()), MALICIOUS_USERS, 'EKS-5: LOF')
pd.DataFrame([h_iso, h_lof])[['Label', 'Recall', 'Precision', 'F1', 'TP', 'FN', 'FP']]

## Informativeness table (not a recall leaderboard)

All tested variants (A/D/E-old/E-v2/EKS-4/EKS-5), focused on alert CHARACTER --
concentration, frequency, analyst load -- not just who wins recall.

In [ ]:
def info_row(df_flag, label, h_eval):
    fl = df_flag['User_ID'].nunique()
    tot_alerts = len(df_flag)
    per_user_weeks = df_flag.groupby('User_ID')['Year_Week'].nunique()
    return {
        'Configuration': label, 'Recall (Answer Set)': h_eval['Recall'],
        'Accounts flagged (unique)': fl, 'Total alerts (account-weeks)': tot_alerts,
        'Alerts per flagged account': round(tot_alerts / fl, 2) if fl else 0,
        'Median flagged-weeks/account': per_user_weeks.median() if fl else 0,
        'Chronic (>=5 weeks) %': round((per_user_weeks >= 5).mean() * 100, 1) if fl else 0,
        '% population flagged': round(fl / df_master['User_ID'].nunique() * 100, 2),
    }

h_D_eval = evaluate_against_answerset(set(df_D['User_ID'].unique()), MALICIOUS_USERS, '')
tabel_informativeness = pd.DataFrame([
    info_row(df_threats, 'A: K-Means->DBSCAN (final design)', h_A),
    info_row(df_D, 'D: K-Means->K-Means', h_D_eval),
    info_row(df_E, 'E-old: DBSCAN->DBSCAN (arbitrary eps2)', h_E_old),
    info_row(df_E_v2, 'E-v2: DBSCAN->DBSCAN (label-free eps2, REJECTED)', h_E_v2),
    info_row(df_iso, 'EKS-4: Isolation Forest', h_iso),
    info_row(df_lof, 'EKS-5: LOF', h_lof),
])
print("Read as: high 'Chronic %' is good ONLY when '% population flagged' is small.")
print("A config with high chronic but flagging tens of percent of the population")
print("(E-v2, IsoForest, LOF) is not a precision detector -- a wide net that")
print("happens to also catch the target. Design A: high chronic on a small flagged")
print("population -- that's what matters to a SOC analyst, not recall alone.")
tabel_informativeness.to_csv('../results/tables/AWIK_Informativeness_Table.csv', index=False)
tabel_informativeness

## Figure: signal persistence by configuration (Section 4.4)

In [ ]:
configs = ['K-Means\nstandalone', 'DBSCAN\nstandalone', 'AWIK\n(two-stage)']
mean_wks = [c2.mean(), c3.mean(), c1.mean()]
med_wks = [c2.median(), c3.median(), c1.median()]
chronic = [(c2>=5).mean()*100, (c3>=5).mean()*100, (c1>=5).mean()*100]

x = np.arange(len(configs)); w = 0.28
c_mean, c_med, c_chronic = '#2196F3', '#FF9800', '#4CAF50'

fig, ax1 = plt.subplots(figsize=(7, 4.5))
ax2 = ax1.twinx()
b1 = ax1.bar(x - w/2, mean_wks, w, color=c_mean, label='Mean wk', edgecolor='white')
b2 = ax1.bar(x + w/2, med_wks, w, color=c_med, label='Median wk', edgecolor='white')
ln = ax2.plot(x, chronic, 'o--', color=c_chronic, lw=2, ms=8, label='Chronic rate (%)', zorder=5)

for bar, val in zip(b1, mean_wks):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.2f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
for bar, val in zip(b2, med_wks):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.1f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
for xi, val in zip(x, chronic):
    ax2.text(xi, val+2.5, f'{val:.1f}%', ha='center', va='bottom', fontsize=8.5, color=c_chronic, fontweight='bold')

rasio = mean_wks[2] / mean_wks[0]
ax1.annotate('', xy=(x[2]-w/2, mean_wks[2]), xytext=(x[0]-w/2, mean_wks[0]),
             arrowprops=dict(arrowstyle='<->', color='#555', lw=1.2))
ax1.text(1.0, (mean_wks[0]+mean_wks[2])/2 + 1.2, f'{rasio:.2f}x', ha='center', fontsize=8.5, color='#555', style='italic')

ax1.set_xticks(x); ax1.set_xticklabels(configs, fontsize=9)
ax1.set_ylabel('Detection-weeks per user', fontsize=9)
ax1.set_ylim(0, max(mean_wks)*1.4)
ax1.grid(axis='y', alpha=0.25); ax1.spines['top'].set_visible(False)
ax2.set_ylabel('Chronic-flagging rate (%)', fontsize=9, color=c_chronic)
ax2.tick_params(axis='y', colors=c_chronic); ax2.set_ylim(0, 100); ax2.spines['top'].set_visible(False)

handles = [b1, b2, ln[0]]
labels = ['Mean detection-weeks', 'Median detection-weeks', 'Chronic-flagging rate (>=5 wk)']
ax1.legend(handles, labels, loc='upper right', fontsize=8, framealpha=0.9)
ax1.set_title('Signal Persistence by Detection Configuration\nCERT r6.2 -- 76 weekly windows, 4,000 accounts', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/AWIK_Persistence_Chart.pdf', bbox_inches='tight', dpi=300)
plt.savefig('../results/figures/AWIK_Persistence_Chart.png', bbox_inches='tight', dpi=300)
plt.show()
print(f"Persistence ratio AWIK/K-Means: {rasio:.2f}x")

In [ ]:
io.save_df(persistence_df, '06_persistence_summary')
for name, series in [('c1_awik', c1), ('c2_kmeans', c2), ('c3_dbscan', c3)]:
    series.rename('weeks').reset_index().to_csv(f'../data/interim/06_{name}.csv', index=False)
print("Checkpoint saved -> data/interim/06_*")